In [ ]:
# Importation des bibliothèques nécessaires pour les deux parties
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import copy
import time

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    PowerTransformer
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Chargement du dataset kc_house_data et affichage de sa forme et des premières lignes
df = pd.read_csv("kc_house_data_NaN.csv")

print("Shape :", df.shape)
display(df.head())

In [ ]:
# Affichage des colonnes, types, valeurs manquantes et doublons
print("Colonnes :")
print(df.columns.tolist())

print("\nTypes des variables :")
display(df.dtypes.to_frame("dtype"))

print("\nValeurs manquantes :")
display(df.isnull().sum().to_frame("missing"))

print("\nDoublons exacts :", df.duplicated().sum())

In [ ]:
# Détection des valeurs manquantes déguisées (ex: "unknown", "NA", "?", etc.)
fake_missing = [
    "unknown", "Unknown", "UNKNOWN",
    "na", "NA",
    "null", "NULL",
    "?",
    "none", "None"
]

for col in df.columns:
    nb_fake = df[col].isin(fake_missing).sum()
    if nb_fake > 0:
        print(f"{col} : {nb_fake} valeurs manquantes cachées")

In [ ]:
# Comptage des zéros dans les colonnes numériques pour identifier les valeurs codées 0
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    zero_count = (df[col] == 0).sum()
    print(f"{col} : {zero_count} zéros")

In [ ]:
# Nettoyage : suppression des colonnes non pertinentes et décomposition de la date en year/month/day
clean_df = df.copy()

colonnes_a_supprimer = []
if "Unnamed: 0" in clean_df.columns:
    colonnes_a_supprimer.append("Unnamed: 0")
if "id" in clean_df.columns:
    colonnes_a_supprimer.append("id")
clean_df = clean_df.drop(columns=colonnes_a_supprimer)

clean_df["date"] = pd.to_datetime(clean_df["date"], format="%Y%m%dT%H%M%S")
clean_df["year"]  = clean_df["date"].dt.year
clean_df["month"] = clean_df["date"].dt.month
clean_df["day"]   = clean_df["date"].dt.day
clean_df = clean_df.drop(columns=["date"])

print("Shape après nettoyage :", clean_df.shape)
print("\nValeurs manquantes :")
print(clean_df.isnull().sum())

In [ ]:
# Statistiques descriptives enrichies avec skewness et kurtosis
desc = clean_df.describe().T
desc["skewness"] = clean_df.skew(numeric_only=True)
desc["kurtosis"] = clean_df.kurtosis(numeric_only=True)
display(desc)

In [ ]:
# Visualisation de la distribution et des boxplots de price et sqft_living
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.histplot(clean_df["price"], bins=60, kde=True, ax=axes[0, 0], color="#89B48D")
axes[0, 0].set_title("Distribution de price")

sns.boxplot(x=clean_df["price"], ax=axes[1, 0], color="#89B48D")
axes[1, 0].set_title("Boxplot de price")

sns.histplot(clean_df["sqft_living"], bins=60, kde=True, ax=axes[0, 1], color="#89B48D")
axes[0, 1].set_title("Distribution de sqft_living")

sns.boxplot(x=clean_df["sqft_living"], ax=axes[1, 1], color="#89B48D")
axes[1, 1].set_title("Boxplot de sqft_living")

plt.tight_layout()
plt.show()

In [ ]:
# Affichage des histogrammes et boxplots pour bedrooms, bathrooms, sqft_living, sqft_lot
cols_to_plot = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot"]

for col in cols_to_plot:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(clean_df[col], bins=60, kde=True, ax=axes[0], color="#89B48D")
    axes[0].set_title(f"Distribution de {col}")
    sns.boxplot(x=clean_df[col], ax=axes[1], color="#89B48D")
    axes[1].set_title(f"Boxplot de {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Affichage de la matrice de corrélation et tri des variables selon leur corrélation avec price
plt.figure(figsize=(20, 15))
corr = clean_df.corr(numeric_only=True)
sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Matrice de corrélation")
plt.show()

corr_price = corr["price"].sort_values(ascending=False)
display(corr_price.to_frame("corr_with_price"))

In [ ]:
# Nuages de points entre les variables les plus corrélées avec price
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13
})

key_vars = ["sqft_living", "grade", "sqft_above", "bathrooms"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, col in enumerate(key_vars):
    axes[i].scatter(clean_df[col], clean_df["price"], alpha=0.15, s=4, color="#89B48D")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Price")
    axes[i].set_title(f"{col} vs Price")
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering : création de 6 nouvelles variables dérivées
clean_df["age_maison"]     = 2015 - clean_df["yr_built"]
clean_df["renovee"]        = (clean_df["yr_renovated"] > 0).astype(int)
clean_df["ratio_surface"]  = clean_df["sqft_living"] / (clean_df["sqft_lot"] + 1)
clean_df["total_rooms"]    = clean_df["bedrooms"] + clean_df["bathrooms"]
clean_df["sqft_per_room"]  = clean_df["sqft_living"] / (clean_df["total_rooms"] + 1)
clean_df["grade_condition"]= clean_df["grade"] * clean_df["condition"]

# Conversion de zipcode en type catégoriel (variable nominale)
clean_df["zipcode"] = clean_df["zipcode"].astype("category")

print("Nouvelles colonnes créées :", ["age_maison","renovee","ratio_surface","total_rooms","sqft_per_room","grade_condition"])

In [ ]:
# Calcul et affichage des corrélations avec price après le feature engineering
corr_after = clean_df.corr(numeric_only=True)
corr_price_after = corr_after["price"].sort_values(ascending=False)
display(corr_price_after.to_frame("corr_with_price"))

In [ ]:
# Définition des groupes de variables et séparation X / y puis split train/test 80-20
target = "price"

positive_features = [
    "sqft_living", "sqft_above", "sqft_basement",
    "sqft_living15", "sqft_lot15", "ratio_surface", "sqft_per_room"
]
geo_features  = ["lat", "long"]
disc_features = [
    "bedrooms", "bathrooms", "floors", "condition", "grade",
    "view", "total_rooms", "age_maison", "grade_condition"
]
bin_features  = ["waterfront", "renovee"]
time_features = ["year", "month"]
cat_features  = ["zipcode"]

features = positive_features + geo_features + disc_features + bin_features + time_features + cat_features

X = clean_df[features].copy()
y = clean_df[target].copy()

print("Nombre de variables explicatives :", len(features))
print("Shape X :", X.shape)
print("Shape y :", y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

In [ ]:
# Initialisation du OneHotEncoder et de la fonction build_preprocessor commune aux deux parties
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_preprocessor(method):
    """
    Construit le ColumnTransformer selon la méthode de scaling choisie.
    method : 'none' | 'minmax' | 'standard' | 'robust' | 'yeo'
    """
    other_numeric_features = geo_features + disc_features + time_features

    if method == "none":
        positive_pipeline      = Pipeline([("imputer", SimpleImputer(strategy="median"))])
        other_numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

    elif method == "minmax":
        positive_pipeline      = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())])
        other_numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())])

    elif method == "standard":
        positive_pipeline      = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        other_numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

    elif method == "robust":
        positive_pipeline      = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())])
        other_numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", RobustScaler())])

    elif method == "yeo":
        positive_pipeline      = Pipeline([("imputer", SimpleImputer(strategy="median")), ("yeo", PowerTransformer(method="yeo-johnson", standardize=True))])
        other_numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

    else:
        raise ValueError("Méthode inconnue.")

    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("positive",      positive_pipeline,      positive_features),
            ("other_numeric", other_numeric_pipeline, other_numeric_features),
            ("binary",        "passthrough",          bin_features),
            ("cat",           cat_pipeline,           cat_features)
        ],
        remainder="drop",
        sparse_threshold=0.0
    )
    return preprocessor

# Dictionnaire des méthodes de scaling à comparer
methods = {
    "Sans_Scaler":   "none",
    "MinMaxScaler":  "minmax",
    "StandardScaler":"standard",
    "RobustScaler":  "robust",
    "Yeo_Johnson":   "yeo"
}

In [ ]:
# ================================================================
# PARTIE 1 — Modèles SANS transformation de la variable cible y
# ================================================================

In [ ]:
# P1 - LinearRegression : évaluation cross-validation 5-fold sans transformation de y
def build_linear_pipeline_p1(method):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", LinearRegression())
    ])

linear_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + LinearRegression ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_linear_pipeline_p1(method_code)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred   = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        rmse_mean = -scores["test_RMSE"].mean()
        r2_mean   =  scores["test_R2"].mean()
        r2_std    =  scores["test_R2"].std()
        linear_results_p1.append({
            "Méthode": method_name, "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "RMSE_CV_mean": round(rmse_mean,2), "R2_test": round(r2_test,4),
            "RMSE_test": round(rmse_test,2), "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        linear_results_p1.append({"Méthode": method_name, "Temps_sec": round(time.time()-start_time,4)})
        print(f"ERREUR → {e}")

df_linear_p1 = pd.DataFrame(linear_results_p1)
display(df_linear_p1)

In [ ]:
# P1 - Visualisation des résultats LinearRegression : R², RMSE et temps d'exécution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_linear_p1["Méthode"]

axes[0].bar(methodes, df_linear_p1['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.85])
for i, v in enumerate(df_linear_p1['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_linear_p1['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_linear_p1['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_linear_p1['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_linear_p1['Temps_sec']): axes[2].text(i, v+0.2, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 1 — LinearRegression (sans transformation y)", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P1 - Ridge (alpha=10) : évaluation cross-validation et test sans transformation de y
def build_ridge_pipeline_p1(method, alpha=10):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", Ridge(alpha=alpha))
    ])

ridge_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
alpha_fixe = 10

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + Ridge alpha={alpha_fixe} ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_ridge_pipeline_p1(method_code, alpha=alpha_fixe)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        rmse_mean = -scores["test_RMSE"].mean()
        r2_mean   =  scores["test_R2"].mean()
        r2_std    =  scores["test_R2"].std()
        ridge_results_p1.append({
            "Méthode": method_name, "Alpha": alpha_fixe,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "RMSE_CV_mean": round(rmse_mean,2), "R2_test": round(r2_test,4),
            "RMSE_test": round(rmse_test,2), "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        ridge_results_p1.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_ridge_p1 = pd.DataFrame(ridge_results_p1)
display(df_ridge_p1)

In [ ]:
# P1 - Visualisation des résultats Ridge : R², RMSE et temps d'exécution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_ridge_p1["Méthode"]

axes[0].bar(methodes, df_ridge_p1['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.85])
for i, v in enumerate(df_ridge_p1['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_ridge_p1['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_ridge_p1['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_ridge_p1['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_ridge_p1['Temps_sec']): axes[2].text(i, v+0.2, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 1 — Ridge (sans transformation y)", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P1 - Lasso (alpha=1000) : évaluation cross-validation et test sans transformation de y
def build_lasso_pipeline_p1(method, alpha=1000):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", Lasso(alpha=alpha, max_iter=20000))
    ])

lasso_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
alpha_fixe = 1000

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + Lasso alpha={alpha_fixe} ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_lasso_pipeline_p1(method_code, alpha=alpha_fixe)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        rmse_mean = -scores["test_RMSE"].mean()
        r2_mean   =  scores["test_R2"].mean()
        r2_std    =  scores["test_R2"].std()
        lasso_results_p1.append({
            "Méthode": method_name, "Alpha": alpha_fixe,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "RMSE_CV_mean": round(rmse_mean,2), "R2_test": round(r2_test,4),
            "RMSE_test": round(rmse_test,2), "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        lasso_results_p1.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_lasso_p1 = pd.DataFrame(lasso_results_p1)
display(df_lasso_p1)

In [ ]:
# P1 - Visualisation des résultats Lasso : R², RMSE et temps d'exécution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_lasso_p1["Méthode"]

axes[0].bar(methodes, df_lasso_p1['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.85])
for i, v in enumerate(df_lasso_p1['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_lasso_p1['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_lasso_p1['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_lasso_p1['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_lasso_p1['Temps_sec']): axes[2].text(i, v+0.2, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 1 — Lasso (sans transformation y)", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P1 - KNN : GridSearchCV puis évaluation test sans transformation de y
def build_knn_pipeline_p1(method):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", KNeighborsRegressor())
    ])

knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 10, 15, 20],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + KNN ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_knn_pipeline_p1(method_code)
        grid = GridSearchCV(pipeline, knn_param_grid, cv=cv,
            scoring="neg_root_mean_squared_error", n_jobs=-1, refit=True, error_score="raise")
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        best_params   = grid.best_params_
        scores = cross_validate(best_pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1)
        best_pipeline.fit(X_train, y_train)
        y_pred    = best_pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        knn_results_p1.append({
            "Méthode": method_name, "Best_Params": best_params,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        knn_results_p1.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_knn_p1 = pd.DataFrame(knn_results_p1)
display(df_knn_p1)

In [ ]:
# P1 - Visualisation des résultats KNN : R², RMSE et temps d'exécution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_knn_p1["Méthode"]

axes[0].bar(methodes, df_knn_p1['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.5, 0.85])
for i, v in enumerate(df_knn_p1['R2_test']): axes[0].text(i, v+0.005, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_knn_p1['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_knn_p1['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_knn_p1['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_knn_p1['Temps_sec']): axes[2].text(i, v+1, f'{v:.0f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 1 — KNN (sans transformation y)", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P1 - Decision Tree (max_depth=8, min_samples_leaf=10) sans transformation de y
def build_tree_pipeline_p1(method):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42))
    ])

tree_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + DecisionTree ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_tree_pipeline_p1(method_code)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        tree_results_p1.append({
            "Méthode": method_name, "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        tree_results_p1.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_tree_p1 = pd.DataFrame(tree_results_p1)
display(df_tree_p1)

In [ ]:
# P1 - Visualisation des résultats Decision Tree : R², RMSE et temps d'exécution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_tree_p1["Méthode"]

axes[0].bar(methodes, df_tree_p1['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.85])
for i, v in enumerate(df_tree_p1['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_tree_p1['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_tree_p1['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_tree_p1['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_tree_p1['Temps_sec']): axes[2].text(i, v+0.05, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 1 — Decision Tree (sans transformation y)", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P1 - MLP avec GridSearchCV sans transformation de y
def build_mlp_pipeline_p1(method):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", MLPRegressor(random_state=42, early_stopping=True, n_iter_no_change=15, max_iter=500))
    ])

mlp_param_grid = {
    "model__hidden_layer_sizes": [(16, 8, 4)],
    "model__activation": ["relu"],
    "model__alpha": [0.001],
    "model__learning_rate_init": [0.001]
}

mlp_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + MLP ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_mlp_pipeline_p1(method_code)
        grid = GridSearchCV(pipeline, mlp_param_grid, cv=cv,
            scoring="neg_root_mean_squared_error", n_jobs=1, refit=True, error_score="raise")
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        best_params   = grid.best_params_
        scores = cross_validate(best_pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=1)
        best_pipeline.fit(X_train, y_train)
        y_pred    = best_pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        mlp_results_p1.append({
            "Méthode": method_name, "Best_Params": best_params,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        mlp_results_p1.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_mlp_p1 = pd.DataFrame(mlp_results_p1)
display(df_mlp_p1)

In [ ]:
# P1 - SVR avec GridSearchCV sans transformation de y
def build_svr_pipeline_p1(method):
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", SVR())
    ])

svr_param_grid_p1 = {
    "model__C": [1, 10, 100],
    "model__epsilon": [1000, 5000, 10000],
    "model__gamma": ["scale", "auto"]
}

svr_results_p1 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + SVR ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_svr_pipeline_p1(method_code)
        grid = GridSearchCV(pipeline, svr_param_grid_p1, cv=cv,
            scoring="neg_root_mean_squared_error", n_jobs=-1, refit=True, error_score="raise")
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        best_params   = grid.best_params_
        scores = cross_validate(best_pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1)
        best_pipeline.fit(X_train, y_train)
        y_pred    = best_pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        svr_results_p1.append({
            "Méthode": method_name, "Best_Params": best_params,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        svr_results_p1.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_svr_p1 = pd.DataFrame(svr_results_p1)
display(df_svr_p1)

In [ ]:
# ================================================================
# PARTIE 2 — Modèles AVEC transformation logarithmique de y (log1p)
#            via TransformedTargetRegressor
# ================================================================

In [ ]:
# P2 - LinearRegression avec TransformedTargetRegressor (log1p sur y)
def build_linear_pipeline_p2(method):
    model = TransformedTargetRegressor(
        regressor=LinearRegression(),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

linear_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + LinearRegression (log y) ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_linear_pipeline_p2(method_code)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        linear_results_p2.append({
            "Méthode": method_name, "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        linear_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_linear_p2 = pd.DataFrame(linear_results_p2)
display(df_linear_p2)

In [ ]:
# P2 - Visualisation des résultats LinearRegression avec log(y) : R², RMSE et temps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_linear_p2["Méthode"]

axes[0].bar(methodes, df_linear_p2['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.90])
for i, v in enumerate(df_linear_p2['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_linear_p2['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_linear_p2['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_linear_p2['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_linear_p2['Temps_sec']): axes[2].text(i, v+0.1, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 2 — LinearRegression (avec log(y))", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P2 - Ridge (alpha=10) avec TransformedTargetRegressor (log1p sur y)
def build_ridge_pipeline_p2(method, alpha=10):
    model = TransformedTargetRegressor(
        regressor=Ridge(alpha=alpha),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

ridge_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
alpha_fixe = 10

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + Ridge (log y) alpha={alpha_fixe} ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_ridge_pipeline_p2(method_code, alpha=alpha_fixe)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        ridge_results_p2.append({
            "Méthode": method_name, "Alpha": alpha_fixe,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        ridge_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_ridge_p2 = pd.DataFrame(ridge_results_p2)
display(df_ridge_p2)

In [ ]:
# P2 - Visualisation des résultats Ridge avec log(y) : R², RMSE et temps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_ridge_p2["Méthode"]

axes[0].bar(methodes, df_ridge_p2['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.90])
for i, v in enumerate(df_ridge_p2['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_ridge_p2['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_ridge_p2['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_ridge_p2['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_ridge_p2['Temps_sec']): axes[2].text(i, v+0.1, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 2 — Ridge (avec log(y))", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P2 - Lasso (alpha=0.0001) avec TransformedTargetRegressor (log1p sur y)
def build_lasso_pipeline_p2(method, alpha=0.0001):
    model = TransformedTargetRegressor(
        regressor=Lasso(alpha=alpha, max_iter=20000),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

lasso_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
alpha_fixe = 0.0001

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + Lasso (log y) alpha={alpha_fixe} ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_lasso_pipeline_p2(method_code, alpha=alpha_fixe)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        lasso_results_p2.append({
            "Méthode": method_name, "Alpha": alpha_fixe,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        lasso_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_lasso_p2 = pd.DataFrame(lasso_results_p2)
display(df_lasso_p2)

In [ ]:
# P2 - Visualisation des résultats Lasso avec log(y) : R², RMSE et temps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_lasso_p2["Méthode"]

axes[0].bar(methodes, df_lasso_p2['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.90])
for i, v in enumerate(df_lasso_p2['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_lasso_p2['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_lasso_p2['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_lasso_p2['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_lasso_p2['Temps_sec']): axes[2].text(i, v+0.1, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 2 — Lasso (avec log(y))", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P2 - KNN avec TransformedTargetRegressor (log1p sur y) et GridSearchCV
def build_knn_pipeline_p2(method):
    model = TransformedTargetRegressor(
        regressor=KNeighborsRegressor(),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

knn_param_grid_p2 = {
    "model__regressor__n_neighbors": [3, 5, 7, 10, 15],
    "model__regressor__weights": ["uniform", "distance"],
    "model__regressor__p": [1, 2]
}

knn_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + KNN (log y) ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_knn_pipeline_p2(method_code)
        grid = GridSearchCV(pipeline, knn_param_grid_p2, cv=cv,
            scoring="r2", refit=True, n_jobs=-1, error_score="raise")
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        best_params   = grid.best_params_
        scores = cross_validate(best_pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1)
        best_pipeline.fit(X_train, y_train)
        y_pred    = best_pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        knn_results_p2.append({
            "Méthode": method_name, "Best_Params": best_params,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        knn_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_knn_p2 = pd.DataFrame(knn_results_p2)
display(df_knn_p2)

In [ ]:
# P2 - Visualisation des résultats KNN avec log(y) : R², RMSE et temps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_knn_p2["Méthode"]

axes[0].bar(methodes, df_knn_p2['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.5, 0.90])
for i, v in enumerate(df_knn_p2['R2_test']): axes[0].text(i, v+0.005, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_knn_p2['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_knn_p2['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_knn_p2['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_knn_p2['Temps_sec']): axes[2].text(i, v+1, f'{v:.0f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 2 — KNN (avec log(y))", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P2 - Decision Tree (max_depth=10) avec TransformedTargetRegressor (log1p sur y)
def build_tree_pipeline_p2(method, max_depth=10):
    model = TransformedTargetRegressor(
        regressor=DecisionTreeRegressor(max_depth=max_depth, random_state=42),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

tree_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
max_depth_fixed = 10

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + DecisionTree (log y) ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_tree_pipeline_p2(method_code, max_depth=max_depth_fixed)
        scores = cross_validate(pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1, error_score="raise")
        pipeline.fit(X_train, y_train)
        y_pred    = pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        tree_results_p2.append({
            "Méthode": method_name, "max_depth": max_depth_fixed,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        tree_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_tree_p2 = pd.DataFrame(tree_results_p2)
display(df_tree_p2)

In [ ]:
# P2 - Visualisation des résultats Decision Tree avec log(y) : R², RMSE et temps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methodes = df_tree_p2["Méthode"]

axes[0].bar(methodes, df_tree_p2['R2_test'], color='blue')
axes[0].set_title('Test R²', fontweight='bold'); axes[0].set_ylim([0.7, 0.90])
for i, v in enumerate(df_tree_p2['R2_test']): axes[0].text(i, v+0.002, f'{v:.4f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(methodes, df_tree_p2['RMSE_test'], color='red')
axes[1].set_title('Test RMSE', fontweight='bold')
for i, v in enumerate(df_tree_p2['RMSE_test']): axes[1].text(i, v+2000, f'{v:,.0f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(methodes, df_tree_p2['Temps_sec'], color='green')
axes[2].set_title('Temps', fontweight='bold')
for i, v in enumerate(df_tree_p2['Temps_sec']): axes[2].text(i, v+0.05, f'{v:.2f}s', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle("Partie 2 — Decision Tree (avec log(y))", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# P2 - MLP avec TransformedTargetRegressor (log1p sur y) et GridSearchCV
def build_mlp_pipeline_p2(method):
    model = TransformedTargetRegressor(
        regressor=MLPRegressor(max_iter=500, random_state=42),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

mlp_param_grid_p2 = {
    "model__regressor__hidden_layer_sizes": [(50,), (100,), (50, 50)],
    "model__regressor__activation": ["relu", "tanh"],
    "model__regressor__alpha": [0.0001, 0.001],
    "model__regressor__learning_rate_init": [0.001, 0.01]
}

mlp_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + MLP (log y) ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_mlp_pipeline_p2(method_code)
        grid = GridSearchCV(pipeline, mlp_param_grid_p2, cv=cv,
            scoring="r2", refit=True, n_jobs=-1, error_score="raise")
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        best_params   = grid.best_params_
        scores = cross_validate(best_pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1)
        best_pipeline.fit(X_train, y_train)
        y_pred    = best_pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        mlp_results_p2.append({
            "Méthode": method_name, "Best_Params": best_params,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        mlp_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_mlp_p2 = pd.DataFrame(mlp_results_p2)
display(df_mlp_p2)

In [ ]:
# P2 - SVR avec TransformedTargetRegressor (log1p sur y) et GridSearchCV
def build_svr_pipeline_p2(method):
    model = TransformedTargetRegressor(
        regressor=SVR(),
        func=np.log1p,
        inverse_func=np.expm1
    )
    return Pipeline([
        ("preprocessor", build_preprocessor(method)),
        ("model", model)
    ])

svr_param_grid_p2 = {
    "model__regressor__kernel": ["rbf"],
    "model__regressor__C": [1, 10, 100],
    "model__regressor__epsilon": [0.1, 0.2, 0.5],
    "model__regressor__gamma": ["scale", "auto"]
}

svr_results_p2 = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)

for method_name, method_code in methods.items():
    print(f"{method_name:20s} + SVR (log y) ...", end=" ")
    start_time = time.time()
    try:
        pipeline = build_svr_pipeline_p2(method_code)
        grid = GridSearchCV(pipeline, svr_param_grid_p2, cv=cv,
            scoring="r2", refit=True, n_jobs=-1, error_score="raise")
        grid.fit(X_train, y_train)
        best_pipeline = grid.best_estimator_
        best_params   = grid.best_params_
        scores = cross_validate(best_pipeline, X_train, y_train, cv=cv,
            scoring={"RMSE":"neg_root_mean_squared_error","MAE":"neg_mean_absolute_error","R2":"r2"},
            n_jobs=-1)
        best_pipeline.fit(X_train, y_train)
        y_pred    = best_pipeline.predict(X_test)
        rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
        r2_test   = r2_score(y_test, y_pred)
        mae_test  = mean_absolute_error(y_test, y_pred)
        elapsed   = time.time() - start_time
        r2_mean   = scores["test_R2"].mean()
        r2_std    = scores["test_R2"].std()
        svr_results_p2.append({
            "Méthode": method_name, "Best_Params": best_params,
            "R2_CV_mean": round(r2_mean,4), "R2_CV_std": round(r2_std,4),
            "R2_test": round(r2_test,4), "RMSE_test": round(rmse_test,2),
            "MAE_test": round(mae_test,2), "Temps_sec": round(elapsed,4)
        })
        print(f"R²_CV={r2_mean:.4f}±{r2_std:.4f} | Test_R²={r2_test:.4f} | RMSE={rmse_test:,.0f} | {elapsed:.2f}s")
    except Exception as e:
        svr_results_p2.append({"Méthode": method_name})
        print(f"ERREUR → {e}")

df_svr_p2 = pd.DataFrame(svr_results_p2)
display(df_svr_p2)

In [ ]:
# Comparaison finale : tableau récapitulatif Partie 1 (sans log y) vs Partie 2 (avec log y) par modèle
summary_rows = []

modeles = [
    ("LinearRegression", df_linear_p1, df_linear_p2),
    ("Ridge",            df_ridge_p1,  df_ridge_p2),
    ("Lasso",            df_lasso_p1,  df_lasso_p2),
    ("KNN",              df_knn_p1,    df_knn_p2),
    ("DecisionTree",     df_tree_p1,   df_tree_p2),
    ("MLP",              df_mlp_p1,    df_mlp_p2),
    ("SVR",              df_svr_p1,    df_svr_p2),
]

for modele_name, df_p1, df_p2 in modeles:
    for _, row in df_p1.iterrows():
        if "R2_test" in row:
            summary_rows.append({
                "Modèle": modele_name,
                "Méthode": row["Méthode"],
                "Partie": "P1 (sans log y)",
                "R2_test": row.get("R2_test", np.nan),
                "RMSE_test": row.get("RMSE_test", np.nan)
            })
    for _, row in df_p2.iterrows():
        if "R2_test" in row:
            summary_rows.append({
                "Modèle": modele_name,
                "Méthode": row["Méthode"],
                "Partie": "P2 (avec log y)",
                "R2_test": row.get("R2_test", np.nan),
                "RMSE_test": row.get("RMSE_test", np.nan)
            })

df_summary = pd.DataFrame(summary_rows)
display(df_summary.sort_values(["Modèle", "Partie", "R2_test"], ascending=[True, True, False]))